In [10]:
pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ---------------------------------------- 0.1/17.7 MB 262.6 kB/s eta 0:01:08
   ------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
pip install PyMySQL

   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   -------- ------------------------------- 10.2/46.7 kB ? eta -:--:--
   ----------------- ---------------------- 20.5/46.7 kB 59.5 kB/s eta 0:00:01
   -------------------------- ------------- 30.7/46.7 kB 100.9 kB/s eta 0:00:01
   ----------------------------------- ---- 41.0/46.7 kB 122.9 kB/s eta 0:00:01
   ---------------------------------------- 46.7/46.7 kB 129.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import json
import sys
import time
from datetime import date
from pathlib import Path
import mysql.connector
import pandas as pd
import requests


In [36]:
dbpass = input("Enter your MySQL password: ")

In [37]:
connection = mysql.connector.connect(host='localhost',
                                         port=3306,
                                         user='root',
                                         password=dbpass,
                                         database='nhl_analytics_hub')

In [39]:
DB_NAME = 'NHL_Analytics_HUB'
if connection.is_connected():
        print(connection.is_connected())
        cursor = connection.cursor()

        # --- SELECT query ---
        ##cursor.execute( f"CREATE DATABASE IF NOT EXISTS {DB_NAME} "
        #    "DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
        #cursor.execute(f"USE {DB_NAME}")

True


In [22]:
CREATE_TEAMS_TABLE = """
CREATE TABLE IF NOT EXISTS teams (
    team_id         INT AUTO_INCREMENT PRIMARY KEY,
    team_abbrev     VARCHAR(10) UNIQUE,
    team_name       VARCHAR(100),
    conference_name VARCHAR(50),
    division_name   VARCHAR(50),
    logo_url        TEXT
)
"""

In [23]:
cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_TEAMS_TABLE)

In [24]:
CREATE_STD_TABLE = """
CREATE TABLE IF NOT EXISTS standings (
    standing_id   INT AUTO_INCREMENT PRIMARY KEY,
    team_id       INT,
    season        VARCHAR(20),
    games_played  INT,
    wins          INT,
    losses        INT,
    ot_losses     INT,
    points        INT,
    goals_for     INT,
    goals_against INT,
    home_wins     INT,
    away_wins     INT,
    streak_type   VARCHAR(20),
    streak_count  INT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
)
"""

cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_STD_TABLE)

In [25]:
CREATE_PLAYERS_TABLE = """
CREATE TABLE IF NOT EXISTS players (
    player_id      BIGINT PRIMARY KEY,
    team_id        INT,
    first_name     VARCHAR(100),
    last_name      VARCHAR(100),
    position       VARCHAR(10),
    jersey_number  INT,
    birth_date     DATE,
    birth_country  VARCHAR(10),
    height_cm      REAL,
    weight_kg      REAL,
    shoots_catches VARCHAR(5),
    headshot_url   TEXT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
)
"""
cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_PLAYERS_TABLE)

In [26]:
CREATE_GAMES_TABLE = """
CREATE TABLE IF NOT EXISTS games (
    game_id      BIGINT PRIMARY KEY,
    season       VARCHAR(20),
    game_type    INT,
    game_date    DATE,
    home_team_id INT,
    away_team_id INT,
    home_score   INT,
    away_score   INT,
    game_state   VARCHAR(20),
    venue_name   VARCHAR(150),
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
)
"""
cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_GAMES_TABLE)

In [27]:
CREATE_GAMES_STATS_TABLE = """
CREATE TABLE IF NOT EXISTS game_stats (
    stat_id       INT AUTO_INCREMENT PRIMARY KEY,
    game_id       BIGINT,
    player_id     BIGINT,
    team_id       INT,
    goals         INT,
    assists       INT,
    points        INT,
    shots_on_goal INT,
    penalty_min   INT,
    toi           VARCHAR(10),
    plus_minus    INT,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id)
)
"""
cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_GAMES_STATS_TABLE)

In [28]:
CREATE_SKATER_SEASON_STATS_TABLE = """
CREATE TABLE IF NOT EXISTS skater_season_stats (
    stat_id      INT AUTO_INCREMENT PRIMARY KEY,
    player_id    BIGINT,
    season       VARCHAR(20),
    team_id      INT,
    games_played INT,
    goals        INT,
    assists      INT,
    points       INT,
    plus_minus   INT,
    penalty_min  INT,
    shots        INT,
    avg_toi      VARCHAR(10),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id)
)
"""
cursor.execute(f"USE {DB_NAME}")
cursor.execute(CREATE_SKATER_SEASON_STATS_TABLE)

In [29]:
CREATE_GOALIE_SEASON_STATS_TABLE = """
CREATE TABLE IF NOT EXISTS goalie_season_stats (
    stat_id           INT AUTO_INCREMENT PRIMARY KEY,
    player_id         BIGINT,
    season            VARCHAR(20),
    team_id           INT,
    games_played      INT,
    wins              INT,
    losses            INT,
    ot_losses         INT,
    save_pct          FLOAT,
    goals_against_avg FLOAT,
    shutouts          INT,
    saves             INT,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id)
)
"""
cursor.execute(f"USE {DB_NAME}")    
cursor.execute(CREATE_GOALIE_SEASON_STATS_TABLE)

In [6]:
#Fetch API Data
import requests
import pandas as pd
endpoint1 = "https://api-web.nhle.com/v1/standings/now"
response1 = requests.get(endpoint1)
print(response1.status_code)
#print(response1.json())
df1 = pd.DataFrame(response1.json()['standings'])
df1 = df1[["teamAbbrev","teamName","conferenceName","divisionName","teamLogo","seasonId","gamesPlayed","wins","losses","otLosses","points","goalFor","goalAgainst","homeWins","roadWins","streakCode","streakCount"]]
#print(df1["teamAbbrev"].str.get("default"))


200


In [7]:
df1["teamAbbrev"] = df1["teamAbbrev"].str.get("default")
df1["teamName"] = df1["teamName"].str.get("default")
df1 = df1.sort_values("teamAbbrev")  
teams_df = df1[["teamAbbrev", "teamName", "conferenceName", "divisionName", "teamLogo"]]
#print(teams_df)

teams_df = teams_df.astype(object).where(teams_df.notna(), None)  


sql = """
INSERT INTO teams (team_abbrev, team_name, conference_name, division_name, logo_url)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_name = VALUES(team_name),
    conference_name = VALUES(conference_name),
    division_name = VALUES(division_name),
    logo_url = VALUES(logo_url)
"""
# Fresh cursor to avoid "Unread result found"
try:
    cursor.close()
except Exception:
    pass

cursor = connection.cursor(buffered=True)

insert_values = [
    tuple(None if pd.isna(v) else v for v in row)
    for row in teams_df[["teamAbbrev", "teamName", "conferenceName", "divisionName", "teamLogo"]]
    .itertuples(index=False, name=None)
]
cursor.execute(f"USE {DB_NAME}") 
cursor.executemany(sql, insert_values)
connection.commit()
print(f"Rows affected: {cursor.rowcount}")
connection.commit()          
print(cursor.rowcount)

Rows affected: 32
32


In [8]:
cursor.execute("SELECT team_id, team_abbrev FROM teams")
team_ids = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}


In [9]:
print("Team IDs:", team_ids)

Team IDs: {'ANA': 33, 'BOS': 34, 'BUF': 35, 'CAR': 36, 'CBJ': 37, 'CGY': 38, 'CHI': 39, 'COL': 40, 'DAL': 41, 'DET': 42, 'EDM': 43, 'FLA': 44, 'LAK': 45, 'MIN': 46, 'MTL': 47, 'NJD': 48, 'NSH': 49, 'NYI': 50, 'NYR': 51, 'OTT': 52, 'PHI': 53, 'PIT': 54, 'SEA': 55, 'SJS': 56, 'STL': 57, 'TBL': 58, 'TOR': 59, 'UTA': 60, 'VAN': 61, 'VGK': 62, 'WPG': 63, 'WSH': 64}


In [10]:
df1["team_id"] = df1["teamAbbrev"].map(team_ids)      
df1["seasonId"] = df1["seasonId"].astype(str)         

# Pick the columns in the same order as the INSERT below
standings_df = df1[["team_id", "seasonId", "gamesPlayed", "wins", "losses", "otLosses",
                    "points", "goalFor", "goalAgainst", "homeWins", "roadWins",
                    "streakCode", "streakCount"]]
standings_df = standings_df.astype(object).where(standings_df.notna(), None)   # NaN -> NULL

# Remove this season's old rows so running again doesn't create duplicates
season = df1["seasonId"].iloc[0]
cursor.execute(f"USE {DB_NAME}") 
cursor.execute("DELETE FROM standings WHERE season = %s", (season,))

# 4. Insert (standing_id is filled in by MySQL)
sql = """
INSERT INTO standings (team_id, season, games_played, wins, losses, ot_losses,
                       points, goals_for, goals_against, home_wins, away_wins,
                       streak_type, streak_count)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
cursor.executemany(sql, standings_df.values.tolist())
connection.commit()
print(cursor.rowcount, "rows inserted")

32 rows inserted


In [11]:
cursor.execute(f"USE {DB_NAME}") 
cursor.execute("SELECT team_id, team_abbrev FROM teams")
teams = cursor.fetchall()

# 2. Call the roster endpoint once per team and collect all players
all_players = []
for team_id, team_abbrev in teams:
    url = f"https://api-web.nhle.com/v1/roster/{team_abbrev}/current"
    response = requests.get(url)
    print(team_abbrev, response.status_code)

    if response.status_code == 200:
        data = response.json()
        # Flatten the three lists into one
        players = data.get("forwards", []) + data.get("defensemen", []) + data.get("goalies", [])
        for p in players:
            p["team_id"] = team_id          # remember which team this player is on
        all_players.extend(players)

    time.sleep(0.5)                          # short delay between requests

df3 = pd.DataFrame(all_players)

# 3. Clean the data
df3["firstName"] = df3["firstName"].str.get("default")
df3["lastName"] = df3["lastName"].str.get("default")
df3["positionCode"] = df3["positionCode"].replace({"L": "LW", "R": "RW"})

players_df = df3[["id", "team_id", "firstName", "lastName", "positionCode", "sweaterNumber",
                  "birthDate", "birthCountry", "heightInCentimeters", "weightInKilograms",
                  "shootsCatches", "headshot"]]
players_df = players_df.drop_duplicates(subset="id")
players_df = players_df.astype(object).where(players_df.notna(), None)    # NaN -> NULL

# 4. Insert (update the player if he already exists, e.g. after a trade)
sql = """
INSERT INTO players (player_id, team_id, first_name, last_name, position, jersey_number,
                     birth_date, birth_country, height_cm, weight_kg, shoots_catches, headshot_url)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id = VALUES(team_id),
    position = VALUES(position),
    jersey_number = VALUES(jersey_number),
    weight_kg = VALUES(weight_kg),
    headshot_url = VALUES(headshot_url)
"""
cursor.executemany(sql, players_df.values.tolist())
connection.commit()
print(len(players_df), "players loaded")

ANA 200
BOS 200
BUF 200
CAR 200
CBJ 200
CGY 200
CHI 200
COL 200
DAL 200
DET 200
EDM 200
FLA 200
LAK 200
MIN 200
MTL 200
NJD 200
NSH 200
NYI 200
NYR 200
OTT 200
PHI 200
PIT 200
SEA 200
SJS 200
STL 200
TBL 200
TOR 200
UTA 200
VAN 200
VGK 200
WPG 200
WSH 200
769 players loaded


In [12]:
all_games = []
for team_id, team_abbrev in teams:
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/20252026"
    response = requests.get(url)
    print(team_abbrev, response.status_code)

    if response.status_code == 200:
        for g in response.json().get("games", []):
            all_games.append({
                "game_id": g.get("id"),
                "season": str(g.get("season")),
                "game_type": g.get("gameType"),
                "game_date": g.get("gameDate"),
                "home_team_id": team_ids.get(g.get("homeTeam", {}).get("abbrev")),
                "away_team_id": team_ids.get(g.get("awayTeam", {}).get("abbrev")),
                "home_score": g.get("homeTeam", {}).get("score"),     # None if not played yet
                "away_score": g.get("awayTeam", {}).get("score"),
                "game_state": g.get("gameState"),
                "venue_name": g.get("venue", {}).get("default"),
            })

    time.sleep(0.5)                          # short delay between requests

df4 = pd.DataFrame(all_games)

# 3. Clean the data
df4 = df4.drop_duplicates(subset="game_id")                 
df4 = df4.astype(object).where(df4.notna(), None)           # NaN -> NULL

# 4. Insert — INSERT IGNORE skips games that are already in the table
sql = """
INSERT IGNORE INTO games (game_id, season, game_type, game_date, home_team_id, away_team_id,
                          home_score, away_score, game_state, venue_name)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
cursor.execute(f"USE {DB_NAME}") 
cursor.executemany(sql, df4.values.tolist())
connection.commit()
print(len(df4), "games loaded")

ANA 200
BOS 200
BUF 200
CAR 200
CBJ 200
CGY 200
CHI 200
COL 200
DAL 200
DET 200
EDM 200
FLA 200
LAK 200
MIN 200
MTL 200
NJD 200
NSH 200
NYI 200
NYR 200
OTT 200
PHI 200
PIT 200
SEA 200
SJS 200
STL 200
TBL 200
TOR 200
UTA 200
VAN 200
VGK 200
WPG 200
WSH 200
1498 games loaded


In [20]:
# 1. Read the JSON file
df5 = pd.read_json(r".\game_stats.json")

# IDs in the file
file_players = set(df5["player_id"])
file_teams = set(df5["team_id"])

# IDs in the database
cursor.execute("SELECT player_id FROM players")
db_players = {row[0] for row in cursor.fetchall()}

cursor.execute("SELECT team_id FROM teams")
db_teams = {row[0] for row in cursor.fetchall()}

# Compare
print("PLAYERS")
print("  in file:       ", len(file_players))
print("  in players table:", len(db_players))
print("  matching:      ", len(file_players & db_players))

print("TEAMS")
print("  in file:       ", sorted(file_teams))
print("  in teams table:", sorted(db_teams))
print("  matching:      ", len(file_teams & db_teams))

# How many rows would pass both checks
ok = df5["player_id"].isin(db_players) & df5["team_id"].isin(db_teams)
print("Rows that can be inserted:", ok.sum(), "of", len(df5))
cursor.execute("SELECT player_id, team_id FROM players")
db_team_of_player = dict(cursor.fetchall())

df5["db_team_id"] = df5["player_id"].map(db_team_of_player)
team_map = (df5.dropna(subset=["db_team_id"])
              .groupby("team_id")["db_team_id"]
              .agg(lambda s: s.mode()[0]))

print(len(team_map), "teams mapped,", team_map.nunique(), "unique")   # should be 32, 32

# 2b. Replace the file's team_id with yours
df5["team_id"] = df5["team_id"].map(team_map).astype(int)
df5 = df5.drop(columns="db_team_id")

# 2c. Nulls -> NULL
df5 = df5.astype(object).where(df5.notna(), None)

PLAYERS
  in file:        710
  in players table: 769
  matching:       618
TEAMS
  in file:        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
  in teams table: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
  matching:       0
Rows that can be inserted: 0 of 47408
32 teams mapped, 32 unique


In [21]:
# 3. Insert in batches of 5000 rows
sql = """
INSERT IGNORE INTO game_stats (stat_id, game_id, player_id, team_id, goals, assists, points,
                               shots_on_goal, penalty_min, toi, plus_minus)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
rows = df5[["stat_id", "game_id", "player_id", "team_id", "goals", "assists", "points",
            "shots_on_goal", "penalty_min", "toi", "plus_minus"]].values.tolist()

for i in range(0, len(rows), 5000):
    cursor.executemany(sql, rows[i:i + 5000])
    connection.commit()

cursor.execute("SELECT COUNT(*) FROM game_stats")
print("Total rows in game_stats:", cursor.fetchone()[0])

Total rows in game_stats: 44263


In [30]:
# 1. Read the file (keep season as text)
cursor.execute("TRUNCATE TABLE goalie_season_stats")
connection.commit()

In [ ]:
# 1. Read the file
df6 = pd.read_json(r".\skater_season_stats.json", dtype={"season": str})
print(df6.shape)                                  

# 2. Clean
df6["season"] = df6["season"].replace("None", None)          
df6["team_id"] = df6["team_id"].map(team_map).astype(int)    
df6 = df6.astype(object).where(df6.notna(), None)            

# 3. Insert
sql = """
INSERT IGNORE INTO skater_season_stats (stat_id, player_id, season, team_id, games_played, goals,
                                        assists, points, plus_minus, penalty_min, shots, avg_toi)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
rows = df6[["stat_id", "player_id", "season", "team_id", "games_played", "goals", "assists",
            "points", "plus_minus", "penalty_min", "shots", "avg_toi"]].values.tolist()
cursor.execute(f"USE {DB_NAME}") 
cursor.executemany(sql, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM skater_season_stats")
print("Total rows in skater_season_stats:", cursor.fetchone()[0])

(714, 12)
Total rows in skater_season_stats: 620


In [33]:
# 1. Read the file (keep season as text)
df7 = pd.read_json(r".\goalie_season_stats.json", dtype={"season": str})
print(df7.shape)                                   # (81, 12)

# 2. Clean
df7["season"] = df7["season"].replace("None", None)          # text "None" -> NULL
df7["team_id"] = df7["team_id"].map(team_map).astype(int)    # file team ids -> your team ids
df7 = df7.astype(object).where(df7.notna(), None)            # NaN -> NULL

# 3. Insert
sql = """
INSERT IGNORE INTO goalie_season_stats (stat_id, player_id, season, team_id, games_played, wins,
                                        losses, ot_losses, save_pct, goals_against_avg, shutouts, saves)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
rows = df7[["stat_id", "player_id", "season", "team_id", "games_played", "wins", "losses",
            "ot_losses", "save_pct", "goals_against_avg", "shutouts", "saves"]].values.tolist()

cursor.execute(f"USE {DB_NAME}")
cursor.executemany(sql, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM goalie_season_stats")
print("Total rows in goalie_season_stats:", cursor.fetchone()[0])

(81, 12)
Total rows in goalie_season_stats: 66


In [35]:
!pip install  streamlit streamlit-option-menu

  Using cached streamlit-1.65.0-py3-none-any.whl.metadata (10 kB)
  Using cached streamlit_option_menu-0.4.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pyarrow-25.0.1-cp311-cp311-win_amd64.whl.metadata (3.0 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached starlette-1.7.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached websockets-17.2-cp311-cp311-win_amd64.whl.metadata (6.5 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
Using cached streamlit-1.65.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
